# Lecture 6: Sequence Processing & Tagging
### NLP Course 2027

---

## Learning Outcomes
- Understand POS tagging and its role in NLP pipelines
- Build NLTK taggers with increasing sophistication
- Apply chunking to extract noun and verb phrases
- Perform Named Entity Recognition with NLTK and spaCy

**Primary Reference:** *NLP with Python* Ch.5 & Ch.7

## 1. Part-of-Speech Tagging

**POS tagging** assigns grammatical categories to each word in a sentence.

```
Sentence:  The  quick  brown  fox  jumps  over  the  lazy  dog
POS tag:   DT   JJ     JJ     NN   VBZ    IN    DT   JJ    NN
```

### Penn Treebank Tagset (most widely used)
| Tag | Meaning | Example |
|-----|---------|--------|
| NN | Noun singular | dog, cat |
| NNS | Noun plural | dogs, cats |
| VB | Verb base | run, eat |
| VBD | Verb past | ran, ate |
| VBG | Verb gerund | running, eating |
| JJ | Adjective | big, fast |
| RB | Adverb | quickly, very |
| DT | Determiner | the, a, an |
| IN | Preposition | in, on, at |
| PRP | Personal pronoun | I, he, she |
| NNP | Proper noun | London, John |

In [1]:
import nltk
from nltk import pos_tag, word_tokenize

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

sentences = [
    'The quick brown fox jumped over the lazy dog.',
    'She can fish or she can swim.',
    'Time flies like an arrow; fruit flies like a banana.'
]

for sent in sentences:
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    print(f'Sentence: {sent}')
    print(f'Tagged:   {tagged}')
    print()

Sentence: The quick brown fox jumped over the lazy dog.
Tagged:   [('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('fox', 'NN'), ('jumped', 'VBD'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN'), ('.', '.')]

Sentence: She can fish or she can swim.
Tagged:   [('She', 'PRP'), ('can', 'MD'), ('fish', 'VB'), ('or', 'CC'), ('she', 'PRP'), ('can', 'MD'), ('swim', 'VB'), ('.', '.')]

Sentence: Time flies like an arrow; fruit flies like a banana.
Tagged:   [('Time', 'NNP'), ('flies', 'NNS'), ('like', 'IN'), ('an', 'DT'), ('arrow', 'NN'), (';', ':'), ('fruit', 'CC'), ('flies', 'NNS'), ('like', 'IN'), ('a', 'DT'), ('banana', 'NN'), ('.', '.')]



In [2]:
from nltk.corpus import treebank
from nltk import DefaultTagger, UnigramTagger, BigramTagger

nltk.download('treebank', quiet=True)

tagged_sents = treebank.tagged_sents()
train_sents = tagged_sents[:3000]
test_sents  = tagged_sents[3000:]
print(f'Train: {len(train_sents)}  Test: {len(test_sents)}')
print(f'Sample: {train_sents[0][:6]}')

Train: 3000  Test: 914
Sample: [('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ')]


In [3]:
# Build progressively smarter taggers
default_tagger = DefaultTagger('NN')  # always guesses NN
unigram_tagger = UnigramTagger(train_sents, backoff=default_tagger)
bigram_tagger  = BigramTagger(train_sents, backoff=unigram_tagger)

for name, tagger in [('Default', default_tagger),
                      ('Unigram', unigram_tagger),
                      ('Bigram',  bigram_tagger)]:
    acc = tagger.evaluate(test_sents)
    print(f'{name} Tagger: {acc:.4f}')

/var/folders/7d/_r5w36tx3l36spkq_4pyynxm0000gn/T/ipykernel_42615/3174966509.py:9: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  acc = tagger.evaluate(test_sents)


Default Tagger: 0.1433
Unigram Tagger: 0.8741
Bigram Tagger: 0.8810


In [4]:
# POS tag a sentence and explain each tag
text = 'Scientists discovered a new species of deep-sea fish near the Mariana Trench.'
tagged = pos_tag(word_tokenize(text))

tag_meaning = {
    'NN':'Noun', 'NNS':'Nouns(pl)', 'NNP':'Proper Noun', 'NNPS':'Proper Nouns(pl)',
    'VBD':'Verb(past)', 'VBN':'Verb(pp)', 'VBZ':'Verb(3sg)', 'VB':'Verb(base)',
    'JJ':'Adjective', 'IN':'Preposition', 'DT':'Determiner', 'CD':'Cardinal Num',
    'RB':'Adverb', 'PRP':'Pronoun'
}
print(f'{"Word":20s} {"Tag":6s} {"Meaning"}')
print('-' * 45)
for word, tag in tagged:
    print(f'{word:20s} {tag:6s} {tag_meaning.get(tag, tag)}')

Word                 Tag    Meaning
---------------------------------------------
Scientists           NNS    Nouns(pl)
discovered           VBD    Verb(past)
a                    DT     Determiner
new                  JJ     Adjective
species              NNS    Nouns(pl)
of                   IN     Preposition
deep-sea             JJ     Adjective
fish                 NN     Noun
near                 IN     Preposition
the                  DT     Determiner
Mariana              NNP    Proper Noun
Trench               NNP    Proper Noun
.                    .      .


## 2. Chunking: Identifying Phrases

**Chunking** groups tagged words into syntactic phrases using regex-like grammar rules.

```
[The quick brown fox]NP  [jumped]VP  [over]PP  [the lazy dog]NP
```

### IOB Notation
```
The    DT   B-NP   ← Begins a Noun Phrase
quick  JJ   I-NP   ← Inside the NP
fox    NN   I-NP
jumped VBD  B-VP   ← Begins a Verb Phrase
```

### Grammar Format
```
NP: {<DT>?<JJ.*>*<NN.*>+}   # optional DT, any adjectives, then noun(s)
PP: {<IN><NP>}               # preposition + noun phrase
VP: {<VB.*><NP|PP|RB>*}     # verb + optional objects
```

In [5]:
from nltk import RegexpParser

grammar = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}
  VP: {<VB.*><NP|PP|RB>*}
  PP: {<IN><NP>}
"""
chunker = RegexpParser(grammar)

sent = 'The brilliant researcher published an important paper about deep learning.'
tagged = pos_tag(word_tokenize(sent))
tree = chunker.parse(tagged)
print('Parse tree:')
print(tree)

Parse tree:
(S
  (NP The/DT brilliant/NN researcher/NN)
  (VP published/VBD (NP an/DT important/JJ paper/NN))
  (PP about/IN (NP deep/JJ learning/NN))
  ./.)


In [6]:
def extract_chunks(text, chunk_type='NP'):
    """Extract all chunks of a given type from text."""
    grammar = r"""
      NP: {<DT>?<JJ.*>*<NN.*>+}
      VP: {<VB.*><RB>?}
    """
    chunker = RegexpParser(grammar)
    chunks = []
    for sent in nltk.sent_tokenize(text):
        tokens = word_tokenize(sent)
        tagged = pos_tag(tokens)
        tree = chunker.parse(tagged)
        for subtree in tree.subtrees():
            if subtree.label() == chunk_type:
                phrase = ' '.join(w for w, t in subtree.leaves())
                chunks.append(phrase)
    return chunks

paragraph = """
The largest language model from OpenAI uses a transformer architecture.
Recent advances in natural language processing have enabled powerful AI systems.
"""
nps = extract_chunks(paragraph, 'NP')
print(f'Extracted {len(nps)} Noun Phrases:')
for np in nps:
    print(f'  - {np}')

Extracted 6 Noun Phrases:
  - The largest language model
  - OpenAI
  - a transformer architecture
  - Recent advances
  - natural language processing
  - powerful AI systems


## 3. Named Entity Recognition

**NER** identifies and classifies proper nouns into semantic categories.

```
'Apple was founded by Steve Jobs in Cupertino, California.'
  ORG                   PERSON      GPE       GPE
```

### Entity Types (NLTK / spaCy)
| Type | Description | Example |
|------|-------------|--------|
| PERSON | People | Barack Obama |
| ORG | Organizations | Google, MIT |
| GPE | Countries, cities, states | France, New York |
| LOC | Non-GPE locations | Mount Everest |
| DATE | Dates/periods | January 2024 |
| MONEY | Monetary values | $1.2 billion |
| PRODUCT | Products | iPhone 15 |

In [7]:
from nltk import ne_chunk
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

def extract_named_entities(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    tree = ne_chunk(tagged, binary=False)
    entities = []
    for subtree in tree:
        if hasattr(subtree, 'label'):
            entity = ' '.join(w for w, t in subtree.leaves())
            entities.append((entity, subtree.label()))
    return entities

news = """Microsoft CEO Satya Nadella announced a partnership with OpenAI.
The deal was worth $10 billion and was signed in Seattle in January 2023.
Google and Meta also have significant AI research divisions."""

for ent, label in extract_named_entities(news):
    print(f'[{label:10s}] {ent}')

[PERSON    ] Microsoft
[ORGANIZATION] CEO Satya Nadella
[ORGANIZATION] OpenAI
[GPE       ] Seattle
[PERSON    ] Google
[ORGANIZATION] Meta


In [8]:
# spaCy: production-grade NER
# !pip install spacy
# !python -m spacy download en_core_web_sm

try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    text = 'Apple was founded by Steve Jobs in Cupertino, California on April 1, 1976.'
    doc = nlp(text)
    print('spaCy Named Entities:')
    for ent in doc.ents:
        print(f'  [{ent.label_:10s}] {ent.text}')
except ImportError:
    print('Install: pip install spacy && python -m spacy download en_core_web_sm')
except OSError:
    print('Model not found. Run: python -m spacy download en_core_web_sm')

Model not found. Run: python -m spacy download en_core_web_sm


In [9]:
# Evaluate NER: simple counting by entity type
def ner_stats(text):
    entities = extract_named_entities(text)
    from collections import Counter
    type_counts = Counter(label for _, label in entities)
    print(f'Total entities: {len(entities)}')
    for etype, count in type_counts.most_common():
        print(f'  {etype}: {count}')
    return entities

article = """
Elon Musk, the CEO of Tesla and SpaceX, announced new products at the San Francisco event.
The company, based in Austin, Texas, reported $25 billion in revenue for 2023.
CFO Zachary Kirkhorn resigned from Tesla in August 2023.
"""
ents = ner_stats(article)

Total entities: 10
  GPE: 5
  ORGANIZATION: 4
  PERSON: 1


## Practice Exercises

See **`Lecture-06-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Task | NLTK Tool | spaCy Tool |
|------|-----------|------------|
| POS Tagging | `pos_tag()` | `token.pos_` |
| Chunking | `RegexpParser` | `noun_chunks` |
| NER | `ne_chunk()` | `doc.ents` |

The pipeline: **tokenize → POS tag → chunk → NER**.

**Next Lecture**: Information Extraction — relations, keyphrases, IE pipelines.

---
*Book references: NLP with Python Ch.5, 7 | Practical NLP Ch.5*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**